In [ ]:
# Lab type: review
# Course: AI401 — AI Applications with LLMs
# Lesson: Validating LLM-Generated Code in Data Pipelines
# Task: Review a sandboxed code runner and audit its security guarantees

# Lab: Reviewing a Sandboxed Code Runner

The implementation below runs LLM-generated Python scripts in a sandboxed subprocess. The code is syntactically correct and will run — your task is to evaluate whether its security guarantees are sufficient for a production pipeline, and to answer judgment questions about what it protects and what it does not.

## Setup

In [ ]:
import subprocess
import tempfile
import json
import hashlib
import os
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime, timezone

## The implementation

In [ ]:
@dataclass
class AuditRecord:
    timestamp: str
    input_hash: str         # SHA-256 of the generated code
    output_summary: str     # first 200 chars of stdout, or error message
    executor: str           # service or job identity
    status: str             # 'ok' | 'error' | 'timeout'
    duration_ms: float


# Credentials to strip from the subprocess environment
_CREDENTIAL_KEYS = {
    'AWS_SECRET_ACCESS_KEY',
    'AWS_ACCESS_KEY_ID',
    'OPENAI_API_KEY',
    'ANTHROPIC_API_KEY',
    'DATABASE_URL',
    'SECRET_KEY',
}


def run_generated_script(
    generated_code: str,
    input_data_path: Path,
    executor: str = 'pipeline-worker',
    timeout_seconds: int = 30,
) -> tuple[dict, AuditRecord]:
    """
    Run generated Python code in a subprocess sandbox.
    Returns (result_dict, audit_record).
    result_dict: {'status': 'ok'|'error'|'timeout', 'output': ..., 'message': ...}
    """
    import time
    t0 = time.perf_counter()
    code_hash = hashlib.sha256(generated_code.encode()).hexdigest()

    with tempfile.TemporaryDirectory() as tmpdir:
        script_path = Path(tmpdir) / 'generated.py'
        output_path = Path(tmpdir) / 'output.json'
        script_path.write_text(generated_code)

        # Strip credentials from environment
        safe_env = {k: v for k, v in os.environ.items()
                    if k not in _CREDENTIAL_KEYS}
        safe_env['INPUT_PATH'] = str(input_data_path)
        safe_env['OUTPUT_PATH'] = str(output_path)

        try:
            result = subprocess.run(
                ['python', str(script_path)],
                env=safe_env,
                capture_output=True,
                text=True,
                timeout=timeout_seconds,
            )
            duration_ms = (time.perf_counter() - t0) * 1000

            if result.returncode != 0:
                msg = result.stderr[:500]
                rec = AuditRecord(
                    timestamp=datetime.now(timezone.utc).isoformat(),
                    input_hash=code_hash[:16],
                    output_summary=msg[:200],
                    executor=executor,
                    status='error',
                    duration_ms=round(duration_ms, 1),
                )
                return {'status': 'error', 'message': msg}, rec

            output = json.loads(output_path.read_text()) if output_path.exists() else {}
            summary = json.dumps(output)[:200]
            rec = AuditRecord(
                timestamp=datetime.now(timezone.utc).isoformat(),
                input_hash=code_hash[:16],
                output_summary=summary,
                executor=executor,
                status='ok',
                duration_ms=round(duration_ms, 1),
            )
            return {'status': 'ok', 'output': output}, rec

        except subprocess.TimeoutExpired:
            duration_ms = (time.perf_counter() - t0) * 1000
            msg = f'Timed out after {timeout_seconds}s'
            rec = AuditRecord(
                timestamp=datetime.now(timezone.utc).isoformat(),
                input_hash=code_hash[:16],
                output_summary=msg,
                executor=executor,
                status='timeout',
                duration_ms=round(duration_ms, 1),
            )
            return {'status': 'error', 'message': msg}, rec

## Inspection cells

Run these before answering the questions below.

In [ ]:
# Inspection 1: run a safe script and inspect the audit record
safe_code = '''
import json, os
output_path = os.environ['OUTPUT_PATH']
result = {'status': 'transformed', 'rows': 42}
with open(output_path, 'w') as f:
    json.dump(result, f)
'''

with tempfile.TemporaryDirectory() as d:
    input_path = Path(d) / 'data.csv'
    input_path.write_text('id,value\n1,100\n2,200')
    result, audit = run_generated_script(safe_code, input_path)

print('Result :', result)
print('Audit  :', audit)

In [ ]:
# Inspection 2: run a script with a runtime error
buggy_code = '''
import pandas as pd  # pandas may not be installed in the sandbox
df = pd.read_csv('nonexistent.csv')
'''

with tempfile.TemporaryDirectory() as d:
    input_path = Path(d) / 'data.csv'
    input_path.write_text('id,value\n1,100')
    result, audit = run_generated_script(buggy_code, input_path)

print('Result :', result)
print('Audit  :', audit)

In [ ]:
# Inspection 3: check that credential env vars are NOT passed to the subprocess
probe_code = '''
import os, json
output_path = os.environ['OUTPUT_PATH']
leaked = {k: v[:4] + '...' for k, v in os.environ.items()
          if k in {'AWS_SECRET_ACCESS_KEY', 'ANTHROPIC_API_KEY', 'DATABASE_URL'}}
json.dump({'leaked_keys': list(leaked.keys())}, open(output_path, 'w'))
'''

with tempfile.TemporaryDirectory() as d:
    input_path = Path(d) / 'data.csv'
    input_path.write_text('id,value\n1,100')
    result, audit = run_generated_script(probe_code, input_path)

print('Leaked credential keys visible to sandbox:', result.get('output', {}).get('leaked_keys', []))
print('(Empty list = credentials stripped correctly)')

## Judgment question 1

> `subprocess.run()` is called **without** `shell=True`. What class of injection attack does this prevent, and how does it work?

> Give a concrete example of a `generated_code` string that would behave differently with `shell=True` vs. without it.

In [ ]:
# Your answer:
#
# shell=True prevents:
#
# Concrete example:
#

<details>
<summary>🔑 Reveal answer — Q1</summary>

**What `shell=False` prevents:** Without `shell=True`, the OS executes the script file directly — no shell interpreter processes the argument string. Shell metacharacters (`; | & $() >`) are passed as literal characters to the program rather than being interpreted as shell operators. This eliminates *shell injection*: an attacker cannot append shell commands to the script path or arguments to execute arbitrary code.

**Concrete example:** With `shell=True`, passing `generated_code = "safe_script.py; rm -rf /tmp"` would execute `safe_script.py` and then `rm -rf /tmp`. Without `shell=True`, the entire string `"safe_script.py; rm -rf /tmp"` is treated as a single filename — the OS looks for a file literally named that, finds nothing, and raises `FileNotFoundError` instead of executing the shell command.

</details>

## Judgment question 2

> When `TimeoutExpired` is caught, the subprocess is left running in the background until the OS kills it (the default). In a high-throughput pipeline, many timed-out processes could accumulate.

> What should be added to the `except subprocess.TimeoutExpired` block, and what `subprocess.run` flag would help terminate the process immediately?

In [ ]:
# Your answer:
#
# What to add:
#
# Relevant subprocess flag:
#

<details>
<summary>🔑 Reveal answer — Q2</summary>

**What to add in `except subprocess.TimeoutExpired`:** Call `proc.kill()` to send SIGKILL immediately, then `proc.communicate()` to drain stdout/stderr and reap the process, preventing zombie accumulation. Without this, the subprocess stays alive until the OS decides to clean it up.

**Relevant `subprocess` flag:** Pass `start_new_session=True` when creating the subprocess. This puts the child in its own process group, so you can kill the entire group (including any grandchildren it spawned) with `os.killpg(os.getpgid(proc.pid), signal.SIGKILL)` — a single `proc.kill()` only kills the direct child and orphans its descendants.

</details>

## Judgment question 3

> `_CREDENTIAL_KEYS` lists 6 specific environment variable names to strip. Name at least 3 categories of sensitive env vars not currently covered, and explain the risk each poses if visible to generated code.

In [ ]:
# Your answer:
#
# Category 1:
#
# Category 2:
#
# Category 3:
#

<details>
<summary>🔑 Reveal answer — Q3</summary>

**Category 1 — Database credentials** (`DATABASE_URL`, `POSTGRES_PASSWORD`, `MYSQL_ROOT_PASSWORD`): generated code could read, modify, or exfiltrate production data directly from the pipeline's own database connection.

**Category 2 — Internal service endpoints and auth tokens** (`REDIS_URL`, `RABBITMQ_URL`, `INTERNAL_API_TOKEN`): generated code could read from or publish to internal message queues and caches, injecting poisoned data into downstream pipeline stages.

**Category 3 — Signing and encryption keys** (`PRIVATE_KEY`, `ENCRYPTION_KEY`, `HMAC_SECRET`, `JWT_SECRET`): generated code could forge authentication tokens, decrypt otherwise-protected payloads, or sign malicious artifacts that downstream systems would accept as trusted.

</details>

## Judgment question 4

> This implementation runs generated code on the **same machine** as the pipeline, in a temporary directory. Describe two concrete ways a malicious generated script could still affect the host machine or pipeline despite the safeguards in place.

In [ ]:
# Your answer:
#
# Attack vector 1:
#
# Attack vector 2:
#

<details>
<summary>🔑 Reveal answer — Q4</summary>

**Attack vector 1 — Arbitrary filesystem writes:** The subprocess runs as the same OS user as the pipeline with no filesystem namespace isolation. Generated code can write to any path that user owns — `~/.bashrc`, cron directories, pipeline config files, or SSH `authorized_keys`. A single write to a startup file establishes persistence that survives the subprocess and the temp directory cleanup.

**Attack vector 2 — Unrestricted outbound network access:** No network policy or namespace prevents the subprocess from making outbound connections. Generated code can exfiltrate the contents of the temp directory (including processed data) via HTTP, query the AWS EC2 instance metadata endpoint (`169.254.169.254`) to retrieve IAM credentials, or download and execute additional malicious payloads before the timeout fires.

</details>